# 03 — Climatology, threshold, anomalies (mhw3d.legacy, fixed baseline)

`legacy.compute_climatology` / `compute_threshold` with `climatologyPeriod=cfg.BASELINE`
(Hobday et al. 2016 defaults: 90th pct, ±5-day pooling, 31-day smoothing). `common.calculate_severity`
then gives daily `T_anom` and `severity` for the full record.

Outputs: `clim_thresh_*.nc`, `anom_severity_daily_*.zarr`, `sst_anom_monthly_*.nc`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ausenso as cfg
warnings.filterwarnings("ignore", category=RuntimeWarning)
cfg.ensure_dirs()
from mhw3d import legacy, common
client = cfg.start_client()
client

In [ ]:
sst = xr.open_zarr(cfg.F_SST).sst
print("baseline:", cfg.BASELINE, "| record:", str(sst.time.values[0])[:10], "→", str(sst.time.values[-1])[:10])
sst_c = sst.chunk({"time": -1, **cfg.CHUNKS_CLIM})
seas = legacy.compute_climatology(sst_c, climatologyPeriod=cfg.BASELINE)
thresh = legacy.compute_threshold(sst_c, climatologyPeriod=cfg.BASELINE)
clim = xr.Dataset({"seas": seas, "thresh": thresh}).compute()
clim.attrs.update(baseline=str(cfg.BASELINE), method="mhw3d.legacy (Hobday 2016)")
clim.to_netcdf(cfg.F_CLIM)
clim

In [ ]:
clim = xr.open_dataset(cfg.F_CLIM)
sev = common.calculate_severity(sst, clim.seas, clim.thresh)
sev = sev[["T_anom", "severity"]].astype("float32").chunk({"time": -1, **cfg.CHUNKS_SPACE})
for v in sev.data_vars:
    sev[v].encoding = {}
sev.attrs.update(baseline=str(cfg.BASELINE))
sev.to_zarr(cfg.F_SEV, mode="w", consolidated=True)

In [ ]:
sev = xr.open_zarr(cfg.F_SEV)
ta = sev.T_anom
nd = ta.isel(lat=0, lon=0).resample(time="MS").count().compute()
complete = nd.time.values[(nd == nd.time.dt.days_in_month).values]
monthly = ta.resample(time="MS").mean().sel(time=complete).compute()
monthly.attrs.update(baseline=str(cfg.BASELINE), units="°C")
monthly.to_dataset(name="T_anom").to_netcdf(cfg.F_SST_MONTHLY)
print("monthly:", str(monthly.time.values[0])[:7], "→", str(monthly.time.values[-1])[:7])

## Checks

In [ ]:
pt = dict(lat=-32.0, lon=115.0, method="nearest")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
clim.seas.sel(**pt).plot(ax=ax[0], label="seas"); clim.thresh.sel(**pt).plot(ax=ax[0], label="thresh")
ax[0].legend(); ax[0].set_title("~32°S 115°E")
masks = xr.open_dataset(cfg.F_MASKS)
cfg.box_mean(monthly, mask=masks.shelf).plot(ax=ax[1]); ax[1].axhline(0, c="k", lw=0.5)
ax[1].set_title("Shelf-mean monthly SST anomaly");
# baseline-period anomaly mean should be ~0
print("mean T_anom over baseline (shelf):",
      float(cfg.box_mean(monthly.sel(time=slice(str(cfg.BASELINE[0]), str(cfg.BASELINE[1]))).mean("time"), mask=masks.shelf)))